In [6]:
%pip install pandas numpy matplot scikit-learn xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 45.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 MB 33.7 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
# LOAD THE DATA
data = pd.read_csv("NYPD_Data.csv")


NEED TO ALSO LOAD THE PRECINT DATASET SO WE CAN MAP GRID CELLS TO THE NEAREST PRECINT.. POTENTIAL CODE BELOW ONCE WE HAVE IT!

To estimate deployment influence, you must transform raw points into a Density Map.
- Define a Grid: Create a bounding box for NYC and divide it into cells 
- Nearest Precinct: Calculate the centroid of each cell and find the Euclidean distance to the nearest Precinct station.
- Map Deployment: Merge your parsed PDF data (officers per precinct) to each grid cell based on its assigned precinct.

In [5]:
# Aggregate unique grid cells
grid_points = data[["grid_id", "lat_grid", "lon_grid"]].drop_duplicates()

# Build nearest-neighbor search
precinct_coords = np.radians(
    precinct_locations[["precinct_lat", "precinct_lon"]]
)

grid_coords = np.radians(
    grid_points[["lat_grid", "lon_grid"]]
)

tree = BallTree(precinct_coords, metric="haversine")
dist, ind = tree.query(grid_coords, k=1)

grid_points["nearest_precinct"] = (
    precinct_locations.iloc[ind.flatten()]["precinct"].values
)

grid_points["estimated_officers"] = (
    precinct_locations.iloc[ind.flatten()]["officers_assigned"].values
)

print(grid_points.head())

KeyError: "None of [Index(['grid_id', 'lat_grid', 'lon_grid'], dtype='str')] are in the [columns]"

In [ ]:
daily_grid = data.groupby(
    ["grid_id", "ARREST_DATE"]
).agg({
    "ARREST_KEY": "count",
    "ARREST_PRECINCT": "first",
    "ARREST_BORO": "first"
}).reset_index()

daily_grid = daily_grid.rename(columns={
    "ARREST_KEY": "crime_count"
})

FOR MODELING TARGET VARIABLE WILL BE CRIME COUNT WITH FEATURES LIKE DEPLOYED FORCES, DAY OF WEEK, MONTH, OTHER TEMPORAL, BOROUGH, PRECINT, LAGGED CRIME (YESTERDAY, 7 DAYS, 30 DAYS)!! RUN A RANDOM FOREST/XGBOOST WITH DEPLOYED FORCES AND WITHOUT
- MAYBE INCLUDE A CONSTRAINT THAT HAS A MINIMUM DEPLOYMENT FLOOR OR PENALTY FOR OVER-CONCENTRATION